EMBED VECTORS WITH OPEN AI, NOT SENTENCETRANSFORMER. TRACK PROGRESS IN uploaded_ids.txt

In [2]:

import openai
import numpy as np
import os
from dotenv import load_dotenv
load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
index_name = os.getenv("PINECONE_INDEX")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [9]:
#need to load environment variables
import os
from dotenv import load_dotenv
from pinecone import Pinecone
load_dotenv(override=True)

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
index_name = os.getenv("PINECONE_INDEX")
#index_name = 'openaicourses'
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(index_name)
# View all vector IDs (up to 100 at a time)
index.describe_index_stats()


{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 7243}},
 'total_vector_count': 7243}

In [10]:
def get_embedding(text, model="text-embedding-3-small"):
    """Generate an embedding for a given text using OpenAI's model."""
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return np.array(response.data[0].embedding)

In [12]:
import pinecone
from pinecone import Pinecone
import pandas as pd
import json
from tqdm import tqdm
import os
import openai
import numpy as np

# --- CONFIG ---
BATCH_SIZE = 100  # upload every 10
SAVE_FILE = "uploaded_ids.txt"
JSON_FILE_PATH = "jsons/v2.json"
MODEL_NAME = "text-embedding-3-small"

# --- SETUP ---
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(index_name)

# Load JSON to DataFrame
with open(JSON_FILE_PATH, "r") as f:
    courses = json.load(f)
df = pd.DataFrame(courses)

# Load uploaded IDs to avoid re-uploading
if os.path.exists(SAVE_FILE):
    with open(SAVE_FILE, "r") as f:
        uploaded_ids = set(line.strip() for line in f)
else:
    uploaded_ids = set()

def build_professor_text(professors):
    if not professors:
        return "No professor information available."

    professor_texts = []
    for prof in professors:
        name = prof.get("name", "Unknown")
        quality = prof.get("quality_rating", "N/A")
        difficulty = prof.get("difficulty", "N/A")
        take_again = prof.get("would_take_again", "N/A")
        num_ratings = prof.get("num_ratings", "N/A")
        department = prof.get("department", "Unknown Dept")
        link = prof.get("profile_link", None)

        prof_summary = (
            f"{name} ({department}) — "
            f"Rating: {quality}/5, Difficulty: {difficulty}/5, "
            f"{take_again} would take again, {num_ratings}"
        )

        if link:
            prof_summary += f" [Profile]({link})"

        professor_texts.append(prof_summary)

    return " | ".join(professor_texts)

def build_embedding_text(metadata):
    return (
        f"{metadata['course_id']}: {metadata['course_name']}\n"
        f"Description: {metadata['description']}\n"
        f"Credits: {metadata['credits']}\n"
        f"Prerequisites: {metadata.get('prerequisites', 'None')}\n"
        f"Professors: {metadata.get('professors', 'No professor information available.')}"
    )

# Get embedding from OpenAI
def get_embedding(text, model=MODEL_NAME):
    response = openai.embeddings.create(
        input=[text],
        model=model
    )
    return np.array(response.data[0].embedding)

# Save progress after batch upload
def save_uploaded_ids(ids):
    with open(SAVE_FILE, "a") as f:
        for cid in ids:
            f.write(cid + "\n")

# --- MAIN INSERTION LOOP ---
def insert_courses_into_pinecone(df):
    batch_vectors = []
    batch_ids = []

    for i, row in tqdm(df.iterrows(), total=len(df), desc="Processing Courses"):
        try:
            # Sanitize ID
            course_id = row.get("course_id", f"NoCourseID_{i}")
            course_id = "".join(c if c.isalnum() else "" for c in course_id)

            # Skip if already uploaded
            # if course_id in uploaded_ids:
            #     continue

            # Build text to embed
            professors = row.get("professors", [])
            if not isinstance(professors, list):
                professors = []

            professor_text = build_professor_text(professors)

            # Metadata
            metadata = {
                "course_id": row['course_id'],
                "course_name": row["course_name"].strip() if row["course_name"] else "No Course Name",
                "description": row["description"],
                "credits": row.get("credits", "N/A"),
                "prerequisites": row.get("prerequisites", "None"),
                "professors": professor_text
            }


            #all the information in the metadata as a string as 'text', for similairty search
            metadata["text"] = build_embedding_text(metadata)

            embedding = get_embedding(metadata["text"])

            # Add to current batch
            batch_vectors.append((course_id, embedding, metadata))
            batch_ids.append(course_id)

            # If batch full, upload
            if len(batch_vectors) >= BATCH_SIZE:
                index.upsert(vectors=batch_vectors)
                save_uploaded_ids(batch_ids)
                print(f"📦 Uploaded batch of {len(batch_vectors)}")
                batch_vectors.clear()
                batch_ids.clear()

        except Exception as e:
            print(f"❌ Error on course {i}: {e}")
            continue

    # Final leftovers
    if batch_vectors:
        index.upsert(vectors=batch_vectors)
        save_uploaded_ids(batch_ids)
        print(f"📦 Uploaded final batch of {len(batch_vectors)}")

    print("✅ Done uploading all courses.")

# Run
insert_courses_into_pinecone(df)


Processing Courses:   1%|▏         | 100/7333 [00:34<1:45:11,  1.15it/s]

📦 Uploaded batch of 100


Processing Courses:   3%|▎         | 200/7333 [01:03<1:01:39,  1.93it/s]

📦 Uploaded batch of 100


Processing Courses:   4%|▍         | 300/7333 [01:31<56:02,  2.09it/s]  

...

📦 Uploaded batch of 100


Processing Courses: 100%|█████████▉| 7301/7333 [37:27<00:14,  2.26it/s]

📦 Uploaded batch of 100


Processing Courses: 100%|██████████| 7333/7333 [37:36<00:00,  3.25it/s]


📦 Uploaded final batch of 33
✅ Done uploading all courses.


In [7]:
df

,course_id,course_name,description,credits,prerequisites,offerings,professors
0,AIP 97,"Academic Internship (2, 4)",Individual placements for field learning. Must...,N/A,"lower-division standing, completion of thirty ...","[fall, winter, spring]",NaN
1,AIP 197,"Academic Internship Program (2, 4, 6, 8, 10, 12)",Individual internship placements integrated wi...,N/A,upper-division standing; department approval.,"[fall, winter, spring]",NaN
2,AIP 197DC,"UCDC: Washington, DC Internship (6, 8, 10)",This internship is attached to the University ...,N/A,upper-division standing; department approval.,[winter],NaN
3,AIP 197P,"Public Service Internship (4, 8, 12)",Individual placements for field learning assoc...,N/A,ninety units minimum completed; 2.5 minimum cu...,[winter],NaN
4,AIP 197T,Academic Internship Program—Special Programs (2),Individual placements for field learning assoc...,2,ninety units minimum completed; 2.5 minimum cu...,[],"[{'name': 'Ryan Hodson', 'quality_rating': '5...."
...,...,...,...,...,...,...,...
7328,WCWP 100,Academic Writing (4),An upper-division workshop course in argumenta...,4,junior/senior standing and must be a Warren Co...,"[fall, winter, spring]","[{'name': 'Keith McCleary', 'quality_rating': ..."
7329,WCWP 160,Technical Writing for Scientists and Engineers...,An upper-division workshop-style writing cours...,4,junior/senior standing.,[],"[{'name': 'Elizabeth Blomstedt', 'quality_rati..."
7330,WARR 189,Academic Mentoring and the Writing Process (2),Students will gain a fundamental understanding...,2,permission of instructor is required to enroll.,[],NaN
7331,WCWP 198,Group Study (2),A directed group study involving research and ...,2,None,[],NaN
